In [1]:
import os
import shutil
import time
from glob import glob
from pathlib import Path

import pandas as pd
import torch
from tqdm import tqdm

In [2]:
def change_file_time(file, t):
    timestamp_str = str(t)
    parsed_time = time.strptime(timestamp_str, '%Y-%m-%d %H:%M:%S')
    ts = time.mktime(parsed_time)
    os.utime(file, (ts, ts))

In [45]:
df = pd.read_csv('wildlife-insights_e28f789b-3c3b-4ad0-9879-e81522823e6a_project-2007160_data/images_2007160.csv')

In [46]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [47]:
len(df['deployment_id'].unique())

31

In [48]:
df = df.sort_values(by='timestamp')

In [49]:
df.reset_index(drop=True, inplace=True)

In [ ]:
# df['location'].to_csv('images_human.csv', index=False, header=False)

```
cat images_human.csv | gsutil cp -I human_images
```

In [50]:
df['deployment_id'] = df['deployment_id'].str.replace('/', '-')

In [51]:
df.head()

,project_id,deployment_id,image_id,filename,sequence_id,location,wi_taxon_id,class,order,family,genus,species,common_name,timestamp,bounding_boxes
0,2007160,cam19 02-10-2024,3cdd309d-d483-4aa4-ad69-a0449986565a,IMG_0306.JPG,8667146,gs://145625598251_2007160_901_pilot_24__main/d...,990ae9dd-7a59-4344-afcb-1b7b21368000,Mammalia,Primates,Hominidae,Homo,sapiens,Human,2020-01-01 00:02:24,"{""{\""detectionBox\"":[0,0.308049142,0.943703771..."
1,2007160,cam19 02-10-2024,03afd9b8-b799-46aa-b691-33804ee9df25,IMG_0316.JPG,8667146,gs://145625598251_2007160_901_pilot_24__main/d...,990ae9dd-7a59-4344-afcb-1b7b21368000,Mammalia,Primates,Hominidae,Homo,sapiens,Human,2020-01-01 00:02:46,"{""{\""detectionBox\"":[0.202122957,0.384211779,0..."
2,2007160,cam19 02-10-2024,61d02a34-ebc0-4be5-8b80-b70a05b8802a,IMG_0321.JPG,8667146,gs://145625598251_2007160_901_pilot_24__main/d...,990ae9dd-7a59-4344-afcb-1b7b21368000,Mammalia,Primates,Hominidae,Homo,sapiens,Human,2020-01-01 00:02:55,NaN
3,2007160,cam19 02-10-2024,2c6e90c2-2efd-402b-ac96-4f69c01bb088,IMG_0328.JPG,8667146,gs://145625598251_2007160_901_pilot_24__main/d...,990ae9dd-7a59-4344-afcb-1b7b21368000,Mammalia,Primates,Hominidae,Homo,sapiens,Human,2020-01-01 00:03:09,NaN
4,2007160,cam19 02-10-2024,e0d45113-cbba-411a-a8bb-7e0c9d21b586,IMG_0335.JPG,8667146,gs://145625598251_2007160_901_pilot_24__main/d...,990ae9dd-7a59-4344-afcb-1b7b21368000,Mammalia,Primates,Hominidae,Homo,sapiens,Human,2020-01-01 00:03:23,NaN


In [60]:
for x in df['deployment_id'].unique():
    Path(f'pilotmt/{x}/person').mkdir(exist_ok=True, parents=True)
    Path(f'pilotmt/{x}/animal').mkdir(exist_ok=True)

In [54]:
preds = pd.read_csv('../yolov5/runs/detect/pilotmt-images/predictions.csv', names=['image', 'pred', 'conf'])

In [55]:
df['image'] = df['location'].apply(lambda x: Path(x).name)

In [56]:
df = df.merge(preds, on='image', how='left')

In [63]:
for x, y, z, t, p in zip(df['location'], df['deployment_id'], df['image'], df['timestamp'], df['pred']):
    if p not in ['person', 'animal'] or not Path(f'images/{z}').exists():
        continue
    out_path = f'pilotmt/{y}/{p}/{z}'
    shutil.copy2(f'images/{z}', out_path)
    change_file_time(out_path, t)